In [1]:
from collections import defaultdict
from pathlib import Path

import ir_datasets
import pandas as pd

In [2]:
DATASET_MAP = {
    "msmarco-passage-trec-dl-2019-judged": "msmarco-passage/trec-dl-2019/judged",
    "msmarco-document-trec-dl-2020-judged": "msmarco-document/trec-dl-2020/judged",
}

In [3]:
qrels = defaultdict(dict)
all_qrels = []
for qrel_path in (Path.cwd().parent / "data").glob("msmarco-passage-trec-dl-*-judged/qrels/*.qrels.txt"):
    qrel = pd.read_csv(
        qrel_path,
        sep=" ",
        header=None,
        names=["query_id", "0", "doc_id", "rel"],
        dtype={"query_id": str, "doc_id": str, "rel": int},
    )
    qrels[qrel_path.parent.parent.stem][qrel_path.name.split(".")[0]] = qrel
    qrel = qrel.copy()
    qrel["dataset"] = qrel_path.parent.parent.stem
    qrel["model"] = qrel_path.name.split(".")[0]
    all_qrels.append(qrel)
all_qrels = pd.concat(all_qrels)
list(qrels)

['msmarco-passage-trec-dl-2020-judged', 'msmarco-passage-trec-dl-2019-judged']

In [4]:
docs = ir_datasets.load("msmarco-passage").docs_store()
queries = pd.concat(
    [
        pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2019/judged").queries_iter()),
        pd.DataFrame(ir_datasets.load("msmarco-passage/trec-dl-2020/judged").queries_iter()),
    ]
).set_index("query_id")["text"]

In [5]:
human_qrels = all_qrels[all_qrels["model"] == "trec"]
model_qrels = all_qrels[all_qrels["model"] != "trec"]
human_qrels

,query_id,0,doc_id,rel,dataset,model
0,23849,q0,1020327,2,msmarco-passage-trec-dl-2020-judged,trec
1,23849,q0,1034183,3,msmarco-passage-trec-dl-2020-judged,trec
2,23849,q0,1120730,0,msmarco-passage-trec-dl-2020-judged,trec
3,23849,q0,1139571,1,msmarco-passage-trec-dl-2020-judged,trec
4,23849,q0,1143724,0,msmarco-passage-trec-dl-2020-judged,trec
...,...,...,...,...,...,...
9255,1133167,q0,8839920,2,msmarco-passage-trec-dl-2019-judged,trec
9256,1133167,q0,8839922,2,msmarco-passage-trec-dl-2019-judged,trec
9257,1133167,q0,944810,0,msmarco-passage-trec-dl-2019-judged,trec
9258,1133167,q0,949411,0,msmarco-passage-trec-dl-2019-judged,trec


In [13]:
std_qrels = human_qrels.copy()
std_qrels = pd.merge(
    std_qrels,
    model_qrels.groupby(["query_id", "doc_id"])["rel"].agg(["mean", "median", "std"]),
    on=["query_id", "doc_id"],
)
std_qrels["diff"] = (std_qrels["rel"] - std_qrels["mean"]).abs()
std_qrels = std_qrels.merge(queries, on="query_id")
std_qrels["doc"] = std_qrels["doc_id"].apply(lambda x: docs.get(x, None).default_text())
std_qrels.sort_values("diff", ascending=False).head(40)

,query_id,0,doc_id,rel,dataset,model,mean,median,std,diff,text,doc
17650,1106007,q0,4173712,0,msmarco-passage-trec-dl-2019-judged,trec,3.000,3.0,0.000000,3.000,define visceral?,Medical Definition of Visceral. Visceral: Refe...
5125,673670,q0,7257306,0,msmarco-passage-trec-dl-2020-judged,trec,3.000,3.0,0.000000,3.000,what is a alm,Application Lifecycle Management (ALM) refers ...
3976,405163,q0,6945011,0,msmarco-passage-trec-dl-2020-judged,trec,3.000,3.0,0.000000,3.000,is caffeine an narcotic,"No, opiates and opioids are derived from poppi..."
17963,1112341,q0,1998330,3,msmarco-passage-trec-dl-2019-judged,trec,0.000,0.0,0.000000,3.000,what is the daily life of thai people,"The baht (Thai: บาท, sign : ฿ ; code: THB) is ..."
13911,183378,q0,1289492,0,msmarco-passage-trec-dl-2019-judged,trec,3.000,3.0,0.000000,3.000,exons definition biology,A portion of DNA that codes for a section of t...
17627,1106007,q0,2960558,0,msmarco-passage-trec-dl-2019-judged,trec,3.000,3.0,0.000000,3.000,define visceral?,Definition of Visceral. Visceral: Referring to...
17924,1112341,q0,1035064,3,msmarco-passage-trec-dl-2019-judged,trec,0.000,0.0,0.000000,3.000,what is the daily life of thai people,Flag of Thailand. Thailand (Kingdom of Thailan...
4612,640502,q0,1091795,0,msmarco-passage-trec-dl-2020-judged,trec,3.000,3.0,0.000000,3.000,what does it mean if your tsh is low,This drop in TSH is an attempt to return circu...
18076,1112341,q0,6303187,3,msmarco-passage-trec-dl-2019-judged,trec,0.000,0.0,0.000000,3.000,what is the daily life of thai people,The currency in Thailand is called the 'Thai B...
4245,555530,q0,6450953,0,msmarco-passage-trec-dl-2020-judged,trec,3.000,3.0,0.000000,3.000,what are best foods to lower cholesterol,"Fish such as salmon, swordfish, tuna and trout..."


In [19]:
std_qrels.groupby(["query_id", "text"])["diff"].mean().sort_values(ascending=False)

query_id  text                                                            
141630    describe how muscles and bones work together to produce movement    1.471875
555530    what are best foods to lower cholesterol                            1.408273
1112341   what is the daily life of thai people                               1.360987
405717    is cdg airport in main paris                                        1.260417
405163    is caffeine an narcotic                                             1.213926
                                                                                ...   
1113256   what is reba mcentire's net worth                                   0.339286
1121709   what are the three percenters?                                      0.296348
855410    what is theraderm used for                                          0.252049
1105792   define: geon                                                        0.247006
1030303   who is aziz hashim                           

In [55]:
from textwrap import wrap
print("\n".join(wrap(std_qrels.sort_values("std", ascending=False).head(10).iloc[1]["doc"])))

Answered by The WikiAnswers® Community. Making the world better, one
answer at a time. The song is called, You've Got A Friend In Me, and
it's performed by Randy Newman & Lyle Lovett. The song is called,
You've Got A Friend In Me, and it's performed by Randy Newman &amp;
Lyle Lovett.


In [35]:
queries.loc["264014"]

'how long is life cycle of flea'